In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from matplotlib.ticker import AutoMinorLocator
from matplotlib.patches import Patch


def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
):
    line_width_pts = stroke_in * 72

    ax.minorticks_on()

    if ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(ticker.LogLocator(subs="auto"))
    else:
        ax.xaxis.set_minor_locator(AutoMinorLocator(minor_ticks_x))

    if ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(ticker.LogLocator(subs="auto"))
    else:
        ax.yaxis.set_minor_locator(AutoMinorLocator(minor_ticks_y))

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color("black")

    ax.tick_params(
        which="major",
        width=line_width_pts,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width_pts * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(label_fontsize)
    ax.yaxis.label.set_size(label_fontsize)

    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontsize(tick_labelsize)

    ax.title.set_fontsize(title_fontsize)


def add_axes_inches(
    fig,
    left_in,
    bottom_in,
    width_in,
    height_in,
    fig_width_in,
    fig_height_in,
):
    return fig.add_axes(
        [
            left_in / fig_width_in,
            bottom_in / fig_height_in,
            width_in / fig_width_in,
            height_in / fig_height_in,
        ]
    )


# ============================================================
# INPUT / OUTPUT STRUCTURE
# ============================================================

INPUT_ROOT = Path("./input").resolve()
OUTPUT_ROOT = Path(
    "./output/fig-s16"
).resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Selected state model used for the implied-timescale panel.
selected_tica_lag = 250
selected_k = 50
chosen_msm_lag = 400

# VAMP model-selection data:
#
# input/
# └── model-selection/
#     └── vamp/
#         ├── t50/
#         ├── t100/
#         ├── t150/
#         ├── t200/
#         └── t250/
#
VAMP_ROOT = (
    INPUT_ROOT
    / "model-selection"
    / "vamp"
)

# Selected state model:
#
# input/
# └── state-model/
#     └── t250_k50/
#         └── bmsm/
#             └── lag400/
#                 └── its-from-P-samples.npz
#
STATE_MODEL_ROOT = (
    INPUT_ROOT
    / "state-model"
    / f"t{selected_tica_lag}_k{selected_k}"
)

BMSM_ROOT = (
    STATE_MODEL_ROOT
    / "bmsm"
    / f"lag{chosen_msm_lag}"
)

ITS_FILE_PATH = (
    BMSM_ROOT
    / "its-from-P-samples.npz"
)

OUTPUT_FILE = (
    OUTPUT_ROOT
    / "its_vamp_scores.pdf"
)


# ============================================================
# TOTAL FIGURE SIZE
# ============================================================

FIG_WIDTH_IN = 6.8
FIG_HEIGHT_IN = 8.7

PANEL_LABEL_X_IN = 0.12

shared_label_fontsize = 10
shared_tick_labelsize = 8
shared_title_fontsize = 9


# ============================================================
# TOP PANEL LAYOUT: VAMP GRID (PANEL A)
# ============================================================

TOP_GRID_PANEL_WIDTH_IN = 2.25
TOP_GRID_PANEL_HEIGHT_IN = 1.25
TOP_GRID_GAP_X_IN = 0.50
TOP_GRID_GAP_Y_IN = 0.60

TOP_GRID_WIDTH_IN = (
    2 * TOP_GRID_PANEL_WIDTH_IN
    + TOP_GRID_GAP_X_IN
)

TOP_GRID_HEIGHT_IN = (
    3 * TOP_GRID_PANEL_HEIGHT_IN
    + 2 * TOP_GRID_GAP_Y_IN
)

TOP_GRID_TOP_Y_IN = 8.50

TOP_GRID_LEFT_IN = (
    FIG_WIDTH_IN - TOP_GRID_WIDTH_IN
) / 2.0

TOP_GRID_BOTTOM_IN = (
    TOP_GRID_TOP_Y_IN
    - TOP_GRID_HEIGHT_IN
)


# ============================================================
# GAP BETWEEN PANELS
# ============================================================

GAP_TOP_TO_BOTTOM_IN = 0.50


# ============================================================
# BOTTOM PANEL LAYOUT: IMPLIED TIMESCALES (PANEL B)
# ============================================================

BOTTOM_ITS_WIDTH_IN = 4.50
BOTTOM_ITS_HEIGHT_IN = 2.50

BOTTOM_ITS_LEFT_IN = (
    FIG_WIDTH_IN
    - BOTTOM_ITS_WIDTH_IN
) / 2.0

BOTTOM_ITS_TOP_Y_IN = (
    TOP_GRID_BOTTOM_IN
    - GAP_TOP_TO_BOTTOM_IN
)

BOTTOM_ITS_BOTTOM_IN = (
    BOTTOM_ITS_TOP_Y_IN
    - BOTTOM_ITS_HEIGHT_IN
)


# ============================================================
# TOP PANEL DATA: VAMP BOXPLOTS
# ============================================================

tica_lags = [
    50,
    100,
    150,
    200,
    250,
]

n_clusters = [
    50,
    75,
    100,
    125,
    150,
]

train_filename = (
    "scores-k-50-75-100-125-150-train.npy"
)

test_filename = (
    "scores-k-50-75-100-125-150-test.npy"
)

train_facecolor = "C0"
test_facecolor = "C1"

box_width = 9.2
box_offset = 5.2
show_fliers = False


# ============================================================
# BOTTOM PANEL DATA: IMPLIED TIMESCALES
# ============================================================

process_count_to_plot = 10
frame_to_ns = 0.1


# ============================================================
# VALIDATE EXPECTED INPUT STRUCTURE
# ============================================================

required_files = []

for lag in tica_lags:
    vamp_lag_directory = (
        VAMP_ROOT
        / f"t{lag}"
    )

    required_files.append(
        vamp_lag_directory
        / train_filename
    )

    required_files.append(
        vamp_lag_directory
        / test_filename
    )

required_files.append(
    ITS_FILE_PATH
)

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]

if missing_files:
    missing_text = "\n".join(
        f"  - {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Required input files are missing:\n"
        f"{missing_text}\n\n"
        "Expected input root:\n"
        f"  {INPUT_ROOT}"
    )

print(
    "Input root:",
    INPUT_ROOT,
)

print(
    "VAMP root:",
    VAMP_ROOT,
)

print(
    "Selected state model:",
    STATE_MODEL_ROOT,
)

print(
    "ITS file:",
    ITS_FILE_PATH,
)


# ============================================================
# LOAD TOP PANEL DATA
# ============================================================

all_train = {}
all_test = {}
all_values = []

for lag in tica_lags:
    vamp_lag_directory = (
        VAMP_ROOT
        / f"t{lag}"
    )

    train_path = (
        vamp_lag_directory
        / train_filename
    )

    test_path = (
        vamp_lag_directory
        / test_filename
    )

    train = np.load(
        train_path
    )

    test = np.load(
        test_path
    )

    if train.ndim < 2:
        raise ValueError(
            f"Expected at least a 2D train array for "
            f"TICA lag {lag}, got shape {train.shape}: "
            f"{train_path}"
        )

    if test.ndim < 2:
        raise ValueError(
            f"Expected at least a 2D test array for "
            f"TICA lag {lag}, got shape {test.shape}: "
            f"{test_path}"
        )

    if train.shape[0] != len(n_clusters):
        raise ValueError(
            f"Train array for TICA lag {lag} has "
            f"shape {train.shape}. Expected axis 0 "
            f"to contain {len(n_clusters)} k values "
            f"{n_clusters}."
        )

    if test.shape[0] != len(n_clusters):
        raise ValueError(
            f"Test array for TICA lag {lag} has "
            f"shape {test.shape}. Expected axis 0 "
            f"to contain {len(n_clusters)} k values "
            f"{n_clusters}."
        )

    all_train[lag] = train
    all_test[lag] = test

    all_values.append(
        np.ravel(train)
    )

    all_values.append(
        np.ravel(test)
    )

    print(
        f"TICA lag {lag:>3}: "
        f"train {train.shape}, "
        f"test {test.shape}"
    )


all_values = np.concatenate(
    all_values
)

finite_values = all_values[
    np.isfinite(all_values)
]

if finite_values.size == 0:
    raise ValueError(
        "No finite VAMP-2 scores were found "
        "in the supplied train/test arrays."
    )

y_min_box = finite_values.min()
y_max_box = finite_values.max()
y_range_box = y_max_box - y_min_box

if y_range_box > 0:
    y_pad_box = (
        0.06
        * y_range_box
    )
else:
    y_pad_box = (
        0.05 * abs(y_max_box)
        if y_max_box != 0
        else 1.0
    )

shared_y_min = (
    y_min_box
    - y_pad_box
)

shared_y_max = (
    y_max_box
    + y_pad_box
)


# ============================================================
# LOAD BOTTOM PANEL DATA
# ============================================================

with np.load(
    ITS_FILE_PATH,
    allow_pickle=True,
) as its_data:

    required_its_keys = [
        "lags",
        "its_mean",
        "its_lower",
        "its_upper",
    ]

    missing_its_keys = [
        key
        for key in required_its_keys
        if key not in its_data.files
    ]

    if missing_its_keys:
        raise KeyError(
            "Missing required keys in "
            f"{ITS_FILE_PATH}:\n"
            + "\n".join(
                f"  - {key}"
                for key in missing_its_keys
            )
        )

    lags = (
        np.asarray(
            its_data["lags"],
            dtype=float,
        )
        * frame_to_ns
    )

    its_mean = (
        np.asarray(
            its_data["its_mean"],
            dtype=float,
        )
        * frame_to_ns
    )

    its_lower = (
        np.asarray(
            its_data["its_lower"],
            dtype=float,
        )
        * frame_to_ns
    )

    its_upper = (
        np.asarray(
            its_data["its_upper"],
            dtype=float,
        )
        * frame_to_ns
    )


# ============================================================
# VALIDATE IMPLIED-TIMESCALE DATA
# ============================================================

if lags.ndim != 1:
    raise ValueError(
        f"'lags' must be 1D, got shape {lags.shape}."
    )

for name, array in [
    ("its_mean", its_mean),
    ("its_lower", its_lower),
    ("its_upper", its_upper),
]:
    if array.ndim != 2:
        raise ValueError(
            f"'{name}' must be 2D, "
            f"got shape {array.shape}."
        )

    if array.shape[0] != len(lags):
        raise ValueError(
            f"'{name}' has {array.shape[0]} lag entries "
            f"but 'lags' contains {len(lags)}."
        )

if (
    its_mean.shape
    != its_lower.shape
    or its_mean.shape
    != its_upper.shape
):
    raise ValueError(
        "its_mean, its_lower, and its_upper "
        "must have identical shapes.\n"
        f"its_mean:  {its_mean.shape}\n"
        f"its_lower: {its_lower.shape}\n"
        f"its_upper: {its_upper.shape}"
    )

if its_mean.shape[1] < process_count_to_plot:
    raise ValueError(
        f"Requested {process_count_to_plot} implied "
        f"timescale processes, but the ITS file "
        f"contains only {its_mean.shape[1]}."
    )

print(
    "ITS data:",
    f"{len(lags)} lag times, "
    f"{its_mean.shape[1]} processes",
)


# ============================================================
# CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH_IN,
        FIG_HEIGHT_IN,
    )
)

fig.text(
    PANEL_LABEL_X_IN / FIG_WIDTH_IN,
    TOP_GRID_TOP_Y_IN / FIG_HEIGHT_IN,
    "A",
    fontsize=14,
    va="top",
    ha="left",
)

fig.text(
    PANEL_LABEL_X_IN / FIG_WIDTH_IN,
    BOTTOM_ITS_TOP_Y_IN / FIG_HEIGHT_IN,
    "B",
    fontsize=14,
    va="top",
    ha="left",
)


# ============================================================
# PANEL A: TOP VAMP GRID
# ============================================================

axes_top = []

for row_index in range(3):
    for col_index in range(2):
        left_in = (
            TOP_GRID_LEFT_IN
            + col_index
            * (
                TOP_GRID_PANEL_WIDTH_IN
                + TOP_GRID_GAP_X_IN
            )
        )

        bottom_in = (
            TOP_GRID_BOTTOM_IN
            + (2 - row_index)
            * (
                TOP_GRID_PANEL_HEIGHT_IN
                + TOP_GRID_GAP_Y_IN
            )
        )

        ax = add_axes_inches(
            fig,
            left_in,
            bottom_in,
            TOP_GRID_PANEL_WIDTH_IN,
            TOP_GRID_PANEL_HEIGHT_IN,
            FIG_WIDTH_IN,
            FIG_HEIGHT_IN,
        )

        axes_top.append(ax)


legend_handles = [
    Patch(
        facecolor=train_facecolor,
        edgecolor="black",
        label="Train",
    ),
    Patch(
        facecolor=test_facecolor,
        edgecolor="black",
        label="Test",
    ),
]


for panel_index, lag in enumerate(
    tica_lags
):
    ax = axes_top[
        panel_index
    ]

    train = all_train[
        lag
    ]

    test = all_test[
        lag
    ]

    train_positions = (
        np.array(
            n_clusters,
            dtype=float,
        )
        - box_offset
    )

    test_positions = (
        np.array(
            n_clusters,
            dtype=float,
        )
        + box_offset
    )

    ax.boxplot(
        [
            train[i]
            for i in range(
                len(n_clusters)
            )
        ],
        positions=train_positions,
        widths=box_width,
        patch_artist=True,
        showfliers=show_fliers,
        boxprops=dict(
            facecolor=train_facecolor,
            edgecolor="black",
            linewidth=1.2,
        ),
        medianprops=dict(
            color="black",
            linewidth=1.2,
        ),
        whiskerprops=dict(
            color="black",
            linewidth=1.2,
        ),
        capprops=dict(
            color="black",
            linewidth=1.2,
        ),
    )

    ax.boxplot(
        [
            test[i]
            for i in range(
                len(n_clusters)
            )
        ],
        positions=test_positions,
        widths=box_width,
        patch_artist=True,
        showfliers=show_fliers,
        boxprops=dict(
            facecolor=test_facecolor,
            edgecolor="black",
            linewidth=1.2,
        ),
        medianprops=dict(
            color="black",
            linewidth=1.2,
        ),
        whiskerprops=dict(
            color="black",
            linewidth=1.2,
        ),
        capprops=dict(
            color="black",
            linewidth=1.2,
        ),
    )

    ax.set_title(
        f"TICA lag = {lag}"
    )

    ax.set_xlabel(
        "k"
    )

    ax.set_ylabel(
        "VAMP-2 score"
    )

    ax.set_xticks(
        n_clusters
    )

    ax.set_xticklabels(
        [
            str(k_value)
            for k_value
            in n_clusters
        ]
    )

    ax.set_xlim(
        min(n_clusters) - 22,
        max(n_clusters) + 22,
    )

    ax.set_ylim(
        shared_y_min,
        shared_y_max,
    )

    format_axes(
        ax,
        minor_ticks_x=2,
        minor_ticks_y=2,
        label_fontsize=shared_label_fontsize,
        tick_labelsize=shared_tick_labelsize,
        title_fontsize=shared_title_fontsize,
        stroke_in=0.01,
    )


legend_ax = axes_top[
    len(tica_lags)
]

legend_ax.set_axis_off()

legend_ax.legend(
    handles=legend_handles,
    loc="center",
    frameon=False,
    fontsize=11,
    ncol=1,
    handlelength=1.6,
)


# ============================================================
# PANEL B: BOTTOM IMPLIED TIMESCALES
# ============================================================

ax_bottom = add_axes_inches(
    fig,
    BOTTOM_ITS_LEFT_IN,
    BOTTOM_ITS_BOTTOM_IN,
    BOTTOM_ITS_WIDTH_IN,
    BOTTOM_ITS_HEIGHT_IN,
    FIG_WIDTH_IN,
    FIG_HEIGHT_IN,
)

ax_bottom.set_xscale(
    "linear"
)

ax_bottom.set_yscale(
    "log"
)

ax_bottom.set_xlim(
    0.0,
    lags.max(),
)

ax_bottom.set_ylim(
    1.0,
    1.0e4,
)

ax_bottom.fill_between(
    lags,
    np.full_like(
        lags,
        1.0,
    ),
    lags,
    color="0.80",
    linewidth=0,
    zorder=0,
)

ax_bottom.plot(
    lags,
    lags,
    linestyle="-",
    color="black",
    linewidth=1.5,
    label="_nolegend_",
    zorder=1,
)

ax_bottom.axvline(
    40.0,
    color="black",
    linestyle="--",
    linewidth=1.5,
    zorder=1,
)

for process_index in range(
    process_count_to_plot
):
    ax_bottom.fill_between(
        lags,
        its_lower[
            :,
            process_index,
        ],
        its_upper[
            :,
            process_index,
        ],
        alpha=0.18,
        linewidth=0,
        zorder=2,
    )

    ax_bottom.plot(
        lags,
        its_mean[
            :,
            process_index,
        ],
        marker="o",
        markersize=4.0,
        linewidth=1.8,
        zorder=3,
    )

ax_bottom.set_xlabel(
    "lag time (ns)"
)

ax_bottom.set_ylabel(
    "timescale (ns)"
)

ax_bottom.xaxis.set_major_locator(
    ticker.MultipleLocator(10)
)

ax_bottom.xaxis.set_minor_locator(
    ticker.MultipleLocator(5)
)

format_axes(
    ax_bottom,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=shared_label_fontsize,
    tick_labelsize=shared_tick_labelsize,
    title_fontsize=shared_title_fontsize,
    stroke_in=0.01,
)

# Restore the desired explicit linear x-axis tick spacing
# after format_axes has configured the generic minor ticks.
ax_bottom.xaxis.set_major_locator(
    ticker.MultipleLocator(10)
)

ax_bottom.xaxis.set_minor_locator(
    ticker.MultipleLocator(5)
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    OUTPUT_FILE,
)

print(
    "Saved figure:",
    OUTPUT_FILE,
)

plt.show()